# Justification des modèles ML par catégorie Oscar

> Document d'**hypothèses de modélisation** pour les 9 catégories retenues, ancré sur le cours *Applied ML for Business* (sessions 01 → 07).
> Chaque section propose **5 modèles candidats**, désigne le **meilleur modèle théorique** et propose un **feature engineering ciblé**.
> Le code de validation empirique sera écrit par un autre membre de l'équipe — ici on **pose les hypothèses**, on ne les confirme pas.

**Auteurs** : Keira (justification) — *suite : Anna / Robin / Jonathan pour l'expérimentation*
**Dataset** : `Data/Processed/oscar_imdb_merged.parquet` — 2427 nominations, 27 cérémonies (2000-2026), 30 colonnes.

---

## 0.1 Catégories cibles

| # | Catégorie (libellé dataset) | n | winners | base-rate |
|---|---|---|---|---|
| 1 | Best Actor in a Leading Role | 135 | 27 | 20.0 % |
| 2 | Best Actress in a Leading Role | 134 | 27 | 20.2 % |
| 3 | Best Actor in a Supporting Role | 128 | 24 | 18.8 % |
| 4 | Best Actress in a Supporting Role | 133 | 26 | 19.5 % |
| 5 | Best Picture | 201 | 27 | **13.4 %** |
| 6 | Best Directing | 128 | 24 | 18.8 % |
| 7 | Best Animated Feature Film | 104 | 24 | 23.1 % |
| 8 | Best Visual Effects | 101 | 26 | **25.7 %** |
| 9 | Best Writing (Original Screenplay) | 131 | 27 | 20.6 % |

**Constats structurants** :
- Petits échantillons (~100–200 lignes par catégorie) → **biais > variance** est le risque dominant ; on doit **régulariser fortement** et préférer les modèles à faible capacité ou les ensembles peu profonds.
- Exactement **un gagnant par (catégorie, année)** → ce n'est pas un problème binaire i.i.d., c'est un **problème de ranking groupé**. La perte logistique est un *proxy* ; l'évaluation doit se faire par `argmax` à l'intérieur de chaque groupe (cf. §3).
- Déséquilibre modéré (13–26 % de gagnants) → pas besoin de SMOTE agressif ; `class_weight='balanced'` ou rien suffit (cf. cours session 02 : ne pas sur-corriger sur petit échantillon).


## 1. Cadre théorique : du problème métier au problème ML

D'après la **session 01 (Problem Formulation)** : "la tâche est dictée par la question métier, pas par la donnée".

**Question métier** : *"Pour une cérémonie à venir, quel nominé va gagner dans cette catégorie ?"*

**Formulation ML retenue** :
- **Cible** : `winner ∈ {0,1}` par nominé.
- **Modèle entraîné** : classifieur binaire qui estime `P(winner=1 | features)`.
- **Décision** : pour chaque `(category, year)` du test, on prend `argmax` des probabilités prédites parmi les nominés du groupe.
- C'est exactement le pattern de la **session 04** (gradient boosting + sortie probabiliste) appliqué avec un post-traitement de ranking.

**Alternative non couverte par le cours** : *learning-to-rank* (XGBRanker / LambdaMART) avec `group=(category, year)`. Plus correcte théoriquement mais hors-périmètre pédagogique → on la mentionne sans la sélectionner.

**Anti-leakage critique** (rappel session 02) :
1. **Pas de fuite temporelle** : les features "historiques" (nb nominations passées de l'acteur, % wins du réalisateur) doivent être calculées **strictement avant** la cérémonie courante. Un acteur nominé en 2018 ne doit pas voir ses nominations 2019–2024 dans ses features.
2. **Pas de fuite de groupe** : un même film apparaît dans plusieurs catégories. Le split doit se faire **par année** (`GroupKFold(groups=year)`), jamais aléatoirement ligne par ligne.
3. **Pas de target encoding non protégé** : si on encode `directors` par taux de win, le calcul doit se faire **par fold** (cf. session 03).


## 2. Menu des modèles enseignés (les seuls candidats légitimes)

Restriction explicite : on ne propose **que des modèles vus en cours** (sessions 02, 04, 05), avec une seule extension naturelle (TF-IDF + LR pour le texte) qui reste un pipeline scikit-learn standard.

| Modèle | Session | Hypothèse implicite | Quand le préférer |
|---|---|---|---|
| **Logistic Regression** (L2 / ElasticNet) | 02, 05 | Frontière linéaire dans l'espace standardisé | Petit échantillon, features bien engineerées, besoin d'interprétabilité forte (coefs) |
| **Decision Tree** (max_depth ≤ 4) | 02, 05 | Règles "si-alors" disjointes | Baseline interprétable, communication aux non-techs ; **rarement champion** |
| **Random Forest** (200 arbres, OOB) | 02, 04, 05 | Moyennage d'arbres décorrélés → réduit variance | Données hétérogènes, features de cardinalité variée, peu de tuning |
| **AdaBoost** (stumps `depth=3`) | 04 | Reweighting séquentiel des exemples difficiles | Signal fortement non linéaire, bruit modéré ; sensible aux outliers |
| **XGBoost** (`max_depth=6`, `lr=0.1`) | 04 | Boosting de gradient, croissance level-wise | Beaucoup de features, interactions complexes, dataset ≥ 1k lignes |
| **LightGBM** (`num_leaves=31-63`) | 04 | Boosting leaf-wise, gain de vitesse | Idem XGBoost + variables catégorielles natives ; **risque overfit** sur petit échantillon |
| **Stacking** (RF + LGBM → LR) | 04 | OOF blending → diversité des erreurs | Quand 2-3 modèles ont des erreurs décorrélées ; coûteux à entraîner |
| *TF-IDF + LR (extension)* | (sklearn standard) | Sac-de-mots linéaire | Catégories où le **synopsis / keywords** porte un signal (Best Picture, Screenplay, Animated) |

**Anti-recommandation** :
- Pas de réseaux de neurones (hors-programme + 100 lignes c'est suicidaire).
- Pas de SVM kernel RBF (non vu, et O(n²) inutile ici).
- Pas de k-NN (sensible à la malédiction de la dimension cf. session 07).


## 3. Stratégie d'évaluation (commune aux 9 catégories)

**Split** :
- `GroupKFold(n_splits=5)` avec `groups=year`. Empêche qu'une année soit éclatée entre train et test (un winner par an doit être *entièrement* dans un fold).
- Variante "production" : split temporel — train = 2000-2020, val = 2021-2023, test = 2024-2026.

**Métriques** (ordre de priorité) :
1. **Top-1 accuracy par groupe** : `1[argmax(P) == winner]` agrégé sur les `(category, year)` du test. C'est la métrique **métier**.
2. **PR-AUC** (Precision-Recall AUC) : plus informative que ROC-AUC sous déséquilibre de classes (rappel session 01 : accuracy seule trompe).
3. **Log-loss** : surveille la calibration (utile si on présente des probabilités à l'utilisateur dans l'app Streamlit).
4. **Macro-F1** binaire : référence du cours, comparable d'une catégorie à l'autre.

**Class imbalance** :
- Activer `class_weight='balanced'` (LR, RF, DT) ou `scale_pos_weight = neg/pos` (XGBoost/LightGBM).
- **Ne pas** appliquer SMOTE : avec 24-27 winners par catégorie, on créerait des synthétiques majoritairement non-représentatifs du test.

**Hyperparamètres** : grille **petite** (≤ 12 combinaisons) en `GridSearchCV(cv=GroupKFold)`, sous peine d'overfitting sur la grille (rappel session 02 : la grille est un *deuxième* set de validation).


## 4. Feature engineering générique (partagé par toutes les catégories)

Inspiré directement de la **session 03** : temporel, ratio, interaction, géographique, catégoriel, binning.

### 4.1 Features "film" (toutes catégories)
| Feature | Calcul | Justification (session 03) |
|---|---|---|
| `log_imdb_votes` | `log1p(imdb_votes)` | Distribution lourde-à-droite ; le log linéarise la relation avec winner pour LR. |
| `log_budget`, `log_revenue` | `log1p(.)` après remplacement des 0 par NaN | Idem ; les valeurs 0 sont des "inconnus", pas des "gratuits". |
| `roi = revenue/budget` | clipper à [0, 50] | Indicateur de succès commercial — anti-corrélé à "film d'auteur primé". |
| `rating_gap` | `imdb_rating - tmdb_vote_average` | Désaccord critique / public (session 03 : ratio/différence). |
| `release_month`, `release_quarter` | parse `release_date` | Les sorties **Q4 (oct-déc)** sont sur-représentées chez les gagnants (théorie du "Oscar bait timing"). |
| `is_late_release` | `release_month >= 10` | Binarisation du timing (binning). |
| `runtime_bin` | `<90, 90-120, 120-150, >150` | Les films > 150 min sont sur-représentés en Best Picture (binning session 03). |
| `n_genres`, `genre_*` (one-hot top-10) | déjà partiellement présent | Cardinalité faible → one-hot direct OK. |
| `is_english`, `is_us_prod` | `original_language=='en'`, `'US' in production_countries` | Biais structurel des Oscars (à documenter dans la présentation, c'est un *fait*, pas un défaut). |
| `is_coproduction` | `len(production_countries) > 1` | Signal qualitatif (films internationaux primés en hausse). |
| `decade` | `(year // 10) * 10` | Capture la dérive temporelle (cf. PSI session 02). |

### 4.2 Features historiques "personne" (catégories acteur/actrice/réalisateur)
Calculées **par nominé**, **strictement avant** l'année courante (anti-leakage critique) :
| Feature | Calcul |
|---|---|
| `n_prior_noms` | nb de nominations toutes catégories avant `year` |
| `n_prior_wins` | nb de victoires avant `year` |
| `years_since_last_nom` | écart à la dernière nomination, NaN si jamais |
| `years_since_last_win` | écart à la dernière victoire |
| `is_first_nomination` | `n_prior_noms == 0` |
| `is_overdue` | `n_prior_noms >= 3 and n_prior_wins == 0` (théorie du "career achievement vote") |

### 4.3 Features texte (catégories où le contenu narratif compte)
| Feature | Calcul | Catégories pertinentes |
|---|---|---|
| `n_keywords` | `len(keywords)` | Toutes (proxy de richesse thématique) |
| `keyword_tfidf` | `TfidfVectorizer(max_features=300)` sur `' '.join(keywords)` | Picture, Screenplay, Animated |
| `overview_tfidf` | TF-IDF sur `overview` (max 500 features, n-grams 1-2) | Screenplay, Picture |
| `tagline_len` | `len(tagline.split())` | Picture (proxy marketing-driven) |

> ⚠ **TF-IDF sur 100 lignes est risqué** : on doit **systématiquement** appliquer une réduction (TruncatedSVD à 20-30 composantes, cf. session 07 PCA-equivalent pour sparse) avant de feeder dans un modèle, ou bien rester en LR + L2 fort.


In [ ]:
# Vérification rapide du dataset (lecture seule, aucune écriture)
import pandas as pd
from pathlib import Path

DATA = Path('../Data/Processed/oscar_imdb_merged.parquet')
df = pd.read_parquet(DATA)

TARGETS = {
    'Best Actor in a Leading Role'        : 'Meilleur Acteur',
    'Best Actress in a Leading Role'      : 'Meilleure Actrice',
    'Best Actor in a Supporting Role'     : 'Meilleur Acteur Second Rôle',
    'Best Actress in a Supporting Role'   : 'Meilleure Actrice Second Rôle',
    'Best Picture'                        : 'Meilleur Film',
    'Best Directing'                      : 'Meilleur Réalisateur',
    'Best Animated Feature Film'          : 'Meilleur Film Animé',
    'Best Visual Effects'                 : 'Meilleurs Effets Visuels',
    'Best Writing (Original Screenplay)'  : 'Meilleur Scénario Original',
}

print(f'Dataset : {df.shape[0]} lignes × {df.shape[1]} colonnes — années {df.year.min()}–{df.year.max()}')
print()
rows = []
for cat_en, cat_fr in TARGETS.items():
    s = df[df.category == cat_en]
    rows.append((cat_fr, cat_en, len(s), int(s.winner.sum()), s.winner.mean()))
summary = pd.DataFrame(rows, columns=['Catégorie (FR)', 'category (dataset)', 'n', 'winners', 'base_rate'])
summary['base_rate'] = summary['base_rate'].map('{:.1%}'.format)
summary


## 5. Meilleur Acteur — *Best Actor in a Leading Role*

**Échantillon** : n = 135, winners = 27, base-rate = 20.0 %.

### 5.1 Cadrage du sous-problème

Classification binaire **par nominé** (1 ligne = 1 acteur nominé une année donnée). Le `nominee_type` est `person` ; `nconst` est disponible → on peut construire un historique propre.
Le signal dominant attendu n'est pas la *qualité absolue* du rôle (impossible à mesurer) mais des **proxies sociologiques** : précédentes nominations, momentum (Golden Globes ne sont pas dans nos données, c'est une limite), réception critique/publique du film porteur, et caractéristiques du rôle (biopic, drame historique).


### 5.2 Signaux théoriques attendus

- **Historique personnel** de l'acteur (`n_prior_noms`, `is_overdue`) — théorie du "vote de carrière".
- **Qualité du film porteur** : `imdb_rating`, `tmdb_vote_average`, `imdb_votes` (popularité).
- **Genre du film** : drames et biopics sur-représentés (>60 % historiquement).
- **Timing de sortie** : `is_late_release` (Q4) est un quasi-prérequis.
- **Co-nominations** : le film est-il aussi nominé Best Picture cette année ? (feature très puissante, à construire en jointure).


### 5.3 Cinq modèles candidats


| # | Modèle | Pour | Contre |
|---|---|---|---|
| 1 | **Logistic Regression L2** (`C=1.0`, `class_weight='balanced'`, standard scaler) | Petit n, interprétation directe des coefs (justifie les hypothèses sociologiques au prof), robuste. | Suppose linéarité → ratera les interactions (ex: "overdue × biopic"). |
| 2 | **Random Forest** (`n_estimators=300`, `max_depth=6`, `min_samples_leaf=5`) | Capte les interactions naturellement, OOB-score "gratuit" (session 04). | Petit n + arbres profonds → overfit ; bias des importances vers high-cardinality (session 05). |
| 3 | **XGBoost** (`max_depth=4`, `lr=0.05`, `n_estimators=400`, early stopping) | Régularisation L1/L2 native, gère missing nativement (utile pour `years_since_last_win`). | Sensible au tuning ; risque overfit grille. |
| 4 | **LightGBM** (`num_leaves=15`, `min_data_in_leaf=10`, `feature_fraction=0.8`) | Catégorielles natives (`genres`), rapide → permet 50 splits CV pour estimer la variance. | Leaf-wise → overfit prononcé sur petit échantillon, à brider sévèrement. |
| 5 | **Stacking** (LR + RF + LGBM → LR meta) | OOF blending corrige le biais individuel (session 04). | Justifié seulement si les 3 base-models ont des erreurs décorrélées — à mesurer avant de stacker. |


### 5.4 Pari théorique : **Logistic Regression L2 (avec features historiques bien construites)**


**Pourquoi** : avec n=135 et un signal majoritairement linéaire (plus l'acteur est "overdue", plus son film est noté, plus la proba grimpe), la **régression logistique régularisée bat systématiquement les ensembles** dans ce régime (rappel session 02 : régime "high bias acceptable, faible variance impérative"). Les coefs servent aussi à *expliquer* la prédiction au jury métier (séance d'éval).

**Plan B** : XGBoost si le feature `is_overdue × is_biopic` (et autres interactions) apporte un gain net mesurable en CV. Plan C : stacking LR+XGBoost si les deux ont des PR-AUC comparables mais des erreurs sur des films différents.


### 5.5 Feature engineering ciblé


| Feature | Recette | Pourquoi (rattaché au cours) |
|---|---|---|
| `n_prior_noms`, `n_prior_wins` | `groupby(nconst)` cumulatif **avec shift** (anti-leakage session 02) | Théorie du vote de carrière ; effet non linéaire → discrétiser en `[0, 1, 2, 3+]` (binning §03). |
| `is_overdue` | `n_prior_noms >= 3 & n_prior_wins == 0` | Interaction binaire = règle métier explicite. |
| `film_is_BP_nominee` | jointure avec catégorie *Best Picture* sur `tconst` et `year` | Signal le plus discriminant historiquement ; **à calculer avec rigueur** (pas de fuite cross-catégorie). |
| `genre_is_biopic` | parse `genres` ∋ {'Biography','History'} | Sur-représentation factuelle des biopics. |
| `runtime_z` | z-score de `runtime_minutes` par décennie | Normalisation décade-par-décade évite la dérive (session 02 PSI). |
| `late_q4_release` | `release_month in {10,11,12}` | Timing Oscar bait. |
| `log_imdb_votes` | `log1p` | Distribution log-normale. |



## 6. Meilleure Actrice — *Best Actress in a Leading Role*

**Échantillon** : n = 134, winners = 27, base-rate = 20.2 %.

### 6.1 Cadrage du sous-problème

Strictement parallèle à Best Actor en structure. Mais l'historique sociologique diffère : la distribution d'âges des gagnantes est bimodale (jeunes premières + vétérantes), le "vote de carrière" joue plus tôt, et les films porteurs sont **moins fréquemment des Best Picture nominees** que côté acteur (asymétrie documentée). Conséquence : la feature `film_is_BP_nominee` est moins puissante, et `n_prior_wins` joue différemment.


### 6.2 Signaux théoriques attendus

- Mêmes signaux qu'acteur, mais **pondérations différentes** attendues.
- **Âge** au moment de la nomination : à dériver de `name.basics.birthYear` (jointure IMDb) — bimodalité.
- **Premier rôle féminin majeur** dans un drame indépendant : sur-représentation des gagnantes "découverte" (ex: Brie Larson, Jennifer Lawrence).


### 6.3 Cinq modèles candidats


| # | Modèle | Justification spécifique |
|---|---|---|
| 1 | **Logistic Regression L2** | Mêmes raisons que Best Actor : petit n, signal majoritairement linéaire. |
| 2 | **Logistic Regression ElasticNet** (`l1_ratio=0.5`) | L1 partielle pour faire de la sélection automatique parmi features historiques et genre — utile si on essaie une douzaine de candidates et qu'on veut un modèle parcimonieux pour la présentation. |
| 3 | **Random Forest** (`max_depth=5`, `min_samples_leaf=5`) | Capte la bimodalité d'âge naturellement (split sur `age < 30` puis `age > 50`). |
| 4 | **XGBoost** (`max_depth=4`, `lr=0.05`) | Idem Best Actor — sait gérer la non-linéarité de l'âge sans engineering manuel. |
| 5 | **Stacking** (LR + RF) | Particulièrement intéressant ici car LR capte le linéaire et RF la bimodalité — erreurs structurellement décorrélées. |


### 6.4 Pari théorique : **Logistic Regression ElasticNet, avec une feature `age_bin` non linéaire**


**Pourquoi** : la bimodalité d'âge est l'écart théorique principal vs. Best Actor. Si on l'encode manuellement en `age_bin ∈ {young, mid, mature}` (binning session 03), la LR retrouve la performance d'un RF tout en restant interprétable. L'ElasticNet ajoute la sélection automatique des proxies historiques.

**Plan B** : RF si la feature `age_bin` artisanale s'avère mal calibrée — laisser l'arbre trouver les seuils tout seul.


### 6.5 Feature engineering ciblé


| Feature | Recette | Pourquoi |
|---|---|---|
| `age_at_ceremony` | jointure `nconst → birthYear` puis `year - birthYear` | Signal théorique fort (bimodalité) absent du dataset actuel. |
| `age_bin` | `pd.cut([0, 30, 45, 100])` | Encode la non-linéarité pour les modèles linéaires (binning session 03). |
| Toutes les features personnes du §4.2 | shift par nconst | Anti-leakage. |
| `film_is_BP_nominee` | jointure cross-cat | Moins puissant que côté acteur mais à garder. |
| `genre_is_drama` / `genre_is_biopic` | parse `genres` | Drame + biopic = quasi-monopole historique. |
| `is_indie` | `budget < median_year` (binning par année) | Sur-représentation des films indépendants gagnants. |



## 7. Meilleur Acteur dans un Second Rôle — *Best Actor in a Supporting Role*

**Échantillon** : n = 128, winners = 24, base-rate = 18.8 %.

### 7.1 Cadrage du sous-problème

Différence structurelle avec les rôles principaux : la catégorie récompense souvent des **performances "showy"** (méchant marquant, rôle court à fort impact), pas la traversée d'un film. Conséquence : `imdb_rating` du film a un **poids relatif plus faible**, et le signal "le film est nominé Best Picture" augmente encore (~70 % des gagnants de cette catégorie viennent d'un BP-nominee).


### 7.2 Signaux théoriques attendus

- **Forte co-nomination Best Picture** du film porteur (encore plus que les leading roles).
- **Présence de stars confirmées** dans des seconds rôles : `is_veteran_in_supporting = (n_prior_noms ≥ 2)`.
- **Genre "ensemble cast"** : drama, crime, war ; films choraux.
- **`n_cast` élevé** du film (films à grande distribution).


### 7.3 Cinq modèles candidats


| # | Modèle | Justification |
|---|---|---|
| 1 | **Logistic Regression L2** | Petit n + signal proche du linéaire ; benchmark. |
| 2 | **Random Forest** (`max_depth=5`) | Capte `film_is_BP_nominee × is_veteran` naturellement. |
| 3 | **AdaBoost** (`DecisionTreeClassifier(max_depth=2)`, 100 stumps) | Le seul cas où AdaBoost vaut le coup d'œil : on a quelques règles très discriminantes (BP-nominee + veteran) que des stumps répétés capturent bien. Si AdaBoost bat RF ici, ça raconte une histoire dans la présentation. |
| 4 | **XGBoost** (`max_depth=4`, early stopping) | Référence boosting moderne. |
| 5 | **Stacking** (LR + AdaBoost + XGBoost → LR) | Diversité d'approches → potentiellement utile si chaque modèle attrape un sous-pattern. |


### 7.4 Pari théorique : **Random Forest (n_estimators=300, max_depth=5, min_samples_leaf=5)**


**Pourquoi** : la catégorie est dominée par 2-3 features très discriminantes (`film_is_BP_nominee`, `n_prior_noms`, `genre_is_drama/crime`) et leurs interactions. Le RF capture ça avec très peu de tuning, le OOB-score donne une validation gratuite (session 04), et l'importance par permutation (session 05) sert à raconter l'histoire.

**Plan B** : AdaBoost si la dynamique "quelques règles fortes" se vérifie ; LR sinon.


### 7.5 Feature engineering ciblé


| Feature | Recette | Pourquoi |
|---|---|---|
| `film_is_BP_nominee` | jointure cross-cat | Le **prédicteur N°1** théorique de cette catégorie. |
| `is_veteran_in_supporting` | `n_prior_noms ≥ 2` (toutes catégories confondues) | Capture le pattern "grande star en second rôle". |
| `is_breakthrough` | `n_prior_noms == 0` | Pattern alternatif "découverte" (Mark Rylance 2016, Mahershala Ali 2017). |
| `film_n_total_noms` | nb de catégories où le film est nominé cette année | Proxy du buzz total autour du film. |
| Features film standard | §4.1 | Genre, rating, votes. |



## 8. Meilleure Actrice dans un Second Rôle — *Best Actress in a Supporting Role*

**Échantillon** : n = 133, winners = 26, base-rate = 19.5 %.

### 8.1 Cadrage du sous-problème

Parallèle à Supporting Actor. Différence principale documentée : la catégorie est **plus volatile** historiquement (proportion de "premières nominations qui gagnent" plus élevée que pour les hommes), donc `is_breakthrough` est plus prédictif que `is_veteran`. Le film porteur est aussi un peu moins systématiquement Best Picture nominee.


### 8.2 Signaux théoriques attendus

- `is_breakthrough` > `is_veteran` (hypothèse à vérifier).
- Sur-représentation des **films indépendants à petit budget** (Lupita Nyong'o, Jennifer Hudson).
- `genre_is_drama` reste dominant ; biopic en seconde position.


### 8.3 Cinq modèles candidats


| # | Modèle | Justification |
|---|---|---|
| 1 | **Logistic Regression L2** | Baseline propre, petit n. |
| 2 | **Decision Tree** (`max_depth=3`) | Interprétabilité maximale ; **utile à la présentation** comme "explication-règle" même s'il ne gagne pas la métrique. |
| 3 | **Random Forest** | Référence pratique. |
| 4 | **XGBoost** | Référence boosting. |
| 5 | **LightGBM** | Variant boosting plus rapide ; à comparer à XGBoost en variance de CV. |


### 8.4 Pari théorique : **Logistic Regression L2 (avec interaction explicite `is_breakthrough × is_indie`)**


**Pourquoi** : si la théorie "breakthrough dans un film indé" est vraie, c'est une interaction simple à 2 variables binaires. La LR avec ce produit explicite battra un RF qui doit la "trouver" sur 130 lignes. Plus interprétable, plus stable.

**Plan B** : RF si l'interaction artisanale n'apparaît pas dans les coefs significatifs.


### 8.5 Feature engineering ciblé


| Feature | Recette | Pourquoi |
|---|---|---|
| `is_breakthrough` | `n_prior_noms == 0` | Pattern dominant hypothétique. |
| `is_indie` | budget dans le tercile bas de l'année | Volet "petit film" du pattern. |
| `breakthrough_in_indie` | `is_breakthrough * is_indie` | Interaction explicite. |
| `age_at_ceremony` | comme pour Best Actress | Distribution d'âges plus jeune que Supporting Actor. |
| Features film standard | §4.1 | |



## 9. Meilleur Film — *Best Picture*

**Échantillon** : n = 201, winners = 27, base-rate = 13.4 %.

### 9.1 Cadrage du sous-problème

**Catégorie la plus riche** (201 lignes, 5 à 10 nominés par année dans la décennie 2010+). C'est la **seule catégorie avec un déséquilibre marqué** (13 % de gagnants) car le nb de nominés a grossi (5→10) sans que le nb de gagnants ne change (1).
Le signal est multidimensionnel : qualité critique, succès commercial, statut "passage obligé culturel", co-nominations dans les autres catégories. C'est la catégorie où le **maximum de features** sont pertinentes simultanément → favorable aux modèles non linéaires capables d'interactions.


### 9.2 Signaux théoriques attendus

- **Nombre total de nominations du film cette année** (`film_n_total_noms`) : le prédicteur historique le plus fort, presque parfait.
- **Genre** : drama domine ; les films d'action/sci-fi gagnent rarement (sauf *LOTR*).
- **Budget moyen** (sweet spot ~30-100M$) : ni petit indé ni blockbuster.
- **Texte** : `keywords` et `overview` portent un signal thématique (drames historiques, war stories).
- **Director track-record** : si le réalisateur a un Oscar ou ≥2 nominations.


### 9.3 Cinq modèles candidats


| # | Modèle | Justification |
|---|---|---|
| 1 | **Logistic Regression L2** | Baseline obligatoire ; performance souvent bluffante si features bien construites. |
| 2 | **Random Forest** (`max_depth=8`, `n_estimators=500`) | Catégorie hétérogène, robuste, OOB pratique. |
| 3 | **XGBoost** (`max_depth=5`, `lr=0.05`, `n_estimators=600`, early stopping) | Plus de données ici (201 lignes) → peut digérer plus de features ; gère bien les interactions complexes. |
| 4 | **LightGBM** (`num_leaves=31`, catégorielles natives) | Pertinent grâce au support natif des catégorielles (genres, langue, production_country) sans one-hot. |
| 5 | **Stacking** (LR + XGBoost + LightGBM → LR) | Catégorie suffisamment riche pour que le stacking apporte un vrai gain ; à coupler avec un base-model **TF-IDF + LR** sur `keywords` pour la diversité. |


### 9.4 Pari théorique : **XGBoost (max_depth=5, early stopping) avec features texte TF-IDF réduites à 20 SVD**


**Pourquoi** : c'est **la catégorie où XGBoost devrait dominer**. Trois raisons :
1. **Volume** (201 lignes, le double des autres) → on sort du régime "régularisation > capacité".
2. **Interactions** : `film_n_total_noms × director_prior_wins × genre`, `late_release × is_drama`, etc., toutes naturellement gérées.
3. **Hétérogénéité des features** : numériques, catégorielles, textuelles (via SVD) → boosting cohabite avec tout.

**Plan B** : Stacking si XGBoost et LR-TFIDF capturent des films différents (à mesurer sur la matrice de confusion par fold). Plan C : LightGBM si le tuning XGBoost coûte trop cher en temps.


### 9.5 Feature engineering ciblé


| Feature | Recette | Pourquoi |
|---|---|---|
| `film_n_total_noms` | `groupby(tconst, year).size()` sur tout le dataset | **Feature reine** : nb de nominations totales du film cette année. |
| `has_acting_nom` | `1` si nominé dans au moins 1 catégorie acting | Composante du buzz. |
| `has_director_nom` | idem pour Best Directing | Forte corrélation avec winning Best Picture. |
| `keywords_tfidf` → TruncatedSVD(20) | TF-IDF + SVD (session 07 PCA-like sur sparse) | Capte les thèmes ("war", "biopic", "racism", "addiction" sur-représentés chez les gagnants). |
| `overview_svd` | TF-IDF(overview) + SVD(10) | Idem pour le pitch. |
| `genre_*` | one-hot top-12 genres | Drame est le genre dominant. |
| `budget_bin` | `<10M, 10-50M, 50-100M, >100M` | Sweet spot non linéaire. |
| `decade` | `(year // 10) * 10` | Le profil-type du gagnant a évolué (drift session 02). |
| `is_english` | `original_language == 'en'` | Réalité statistique des Oscars (à documenter explicitement). |



## 10. Meilleur Réalisateur — *Best Directing*

**Échantillon** : n = 128, winners = 24, base-rate = 18.8 %.

### 10.1 Cadrage du sous-problème

Catégorie **hautement corrélée à Best Picture** historiquement (~70 % des Best Directors gagnent aussi BP la même année, modulo "split" récents type 2019 Cuaron/Green Book).
Le signal réalisateur dépasse celui du film : `n_prior_wins` du réalisateur, status d'auteur. C'est la catégorie où la feature historique personne est **la plus prédictive de toutes**.


### 10.2 Signaux théoriques attendus

- **Historique du réalisateur** : `dir_prior_noms`, `dir_prior_wins`, `dir_years_active`.
- **Le film est aussi nominé Best Picture** (`film_is_BP_nominee`).
- **Genre épique / drame historique / war** : sur-représentation.
- **Runtime > 150 min** : sur-représentation (cinéastes "auteur").
- **Premier film d'un réalisateur** : pattern minoritaire mais marquant (Damien Chazelle, Jordan Peele).


### 10.3 Cinq modèles candidats


| # | Modèle | Justification |
|---|---|---|
| 1 | **Logistic Regression L2** | Si les features historiques réalisateur sont bien faites, c'est un excellent benchmark. |
| 2 | **Random Forest** (`max_depth=6`) | Capte le pattern bimodal "vétéran vs prodige". |
| 3 | **XGBoost** (`max_depth=4`, `lr=0.05`) | Boosting de référence. |
| 4 | **LightGBM** | Avec encodage natif des `directors` (catégorielle high-cardinality). |
| 5 | **Stacking** (LR + RF → LR meta) | LR capture le linéaire historique, RF la bimodalité. |


### 10.4 Pari théorique : **LightGBM avec `directors` en feature catégorielle native + features historiques**


**Pourquoi** : `directors` est une variable **catégorielle de haute cardinalité** (~120 réalisateurs uniques). One-hot exploserait l'espace de features pour 128 lignes → catastrophe. LightGBM gère ça nativement via son `categorical_feature` (cf. session 04 : c'est l'avantage différenciant vs XGBoost). Cela capture l'effet "Spielberg/Scorsese/Bigelow" sans inflation dimensionnelle.

**Plan B** : LR avec **target encoding K-fold** de `directors` (session 03 : target encoding doit être par-fold pour éviter la fuite).


### 10.5 Feature engineering ciblé


| Feature | Recette | Pourquoi |
|---|---|---|
| `dir_prior_noms` | `groupby(directors).cumcount().shift(1)` par année | Signal historique le plus fort. |
| `dir_prior_wins` | similaire pour winners | Auteur consacré. |
| `dir_years_active` | `year - min(year by director)` | Maturité de carrière. |
| `directors_cat` | passer en `category` dtype, `categorical_feature` LightGBM | Encodage natif sans one-hot. |
| `film_is_BP_nominee` | jointure cross-cat | Très fort prédicteur (BP/Director split rare). |
| `runtime_bin_long` | `runtime > 150` | Cinéastes "auteur". |
| `genre_is_war_or_historical` | parse `genres` | Sur-représentation. |



## 11. Meilleur Film Animé — *Best Animated Feature Film*

**Échantillon** : n = 104, winners = 24, base-rate = 23.1 %.

### 11.1 Cadrage du sous-problème

**Catégorie la plus restreinte en nombre** (104 lignes, créée en 2002 → seulement 24 cérémonies). Mais la plus **structurellement déterministe** : Pixar/Disney/DreamWorks/Ghibli dominent à un point qu'une règle "studio = Pixar et année ≤ 2018" donnerait déjà ~50 % d'accuracy.
Le signal "studio de production" est primordial. La feature `production_countries` est moins utile (US écrasant) mais `production_companies` (à récupérer via TMDb — pas dans le dataset actuel !) le serait beaucoup. **Recommandation FE majeure** : enrichir TMDb avec `production_companies`.


### 11.2 Signaux théoriques attendus

- **Studio de production** (Pixar > Disney > DreamWorks > Ghibli > autres).
- **Box-office** : sur-représenté chez les gagnants.
- **Note IMDb** : seuil ~8.0 quasi-éliminatoire.
- **Originalité** : suites moins primées que originaux (à dériver via `keywords` ou détection "2" / "II" dans le titre).
- **Animation 2D vs 3D** : transition historique 2D→3D dans les années 2000.


### 11.3 Cinq modèles candidats


| # | Modèle | Justification |
|---|---|---|
| 1 | **Decision Tree** (`max_depth=4`) | Catégorie où une **règle simple** capture 60 % du signal — l'arbre est aussi performant qu'un ensemble et infiniment plus pédagogique. |
| 2 | **Random Forest** (`max_depth=5`, `n_estimators=200`) | Robustesse + OOB sans surcoût. |
| 3 | **Logistic Regression L2** | Baseline minimaliste. |
| 4 | **XGBoost** (très bridé : `max_depth=3`, `n_estimators=200`) | Référence boosting, à brider à cause de n=104. |
| 5 | **LightGBM** avec `production_companies` en catégoriel | Le seul intérêt vs XGBoost : encoder le studio nativement. |


### 11.4 Pari théorique : **Random Forest (max_depth=5, n_estimators=200, min_samples_leaf=4)**


**Pourquoi** : avec n=104 et un signal dominé par 2-3 features fortes (`studio`, `box_office`, `rating`), un RF léger fait le job sans risquer l'overfit d'un boosting agressif. L'OOB score (session 04) donne une validation interne gratuite, précieuse à si peu de données. Si on ajoute `production_company` en categorical-encoded, LightGBM peut le dépasser.

**Plan B** : Decision Tree `max_depth=4` pour la présentation — *l'arbre lui-même* explique la prédiction. Plan C : LightGBM si `production_company` est récupérée.


### 11.5 Feature engineering ciblé


| Feature | Recette | Pourquoi |
|---|---|---|
| `production_company` ⚠ | **À récupérer via TMDb** (champ disponible dans `/movie/{id}`, pas dans notre cache actuel) | **Feature manquante critique** — à ajouter au notebook EDA_merge. |
| `is_pixar`, `is_disney`, `is_dreamworks`, `is_ghibli` | one-hot des top studios | Capture la domination structurelle. |
| `log_revenue` | `log1p(revenue)` | Box-office signal. |
| `is_sequel` | détection regex titre ou keyword "sequel" | Suites moins primées. |
| `imdb_rating` brut | direct | Seuil ~8 quasi-éliminatoire. |
| `release_decade` | décennie | Capture la dérive 2D→3D. |



## 12. Meilleurs Effets Visuels — *Best Visual Effects*

**Échantillon** : n = 101, winners = 26, base-rate = 25.7 %.

### 12.1 Cadrage du sous-problème

**Catégorie la moins déséquilibrée** (25.7 % de winners). 3-5 nominés par an seulement, donc base-rate élevé. Le signal est **moins sociologique, plus matériel** : **budget** est probablement le prédicteur N°1 (les VFX coûtent cher, donc les budgets élevés gagnent), suivi de genre (sci-fi, fantasy, action). Très peu de signal "historique personne".


### 12.2 Signaux théoriques attendus

- **`log_budget`** : prédicteur dominant attendu (les VFX-heavy films ont des budgets > 100M$).
- **Genre** : sci-fi, fantasy, action, adventure → ~90 % des nominés.
- **`revenue`** : corrélé au budget mais peut diverger (un VFX réussi mais flop commercial est rare).
- **Studio** : Disney/Marvel/Warner sur-représentés.
- **Innovation technique** : difficile à mesurer ; proxy = `is_first_in_franchise` (Avatar 2009 > Avatar 2022).


### 12.3 Cinq modèles candidats


| # | Modèle | Justification |
|---|---|---|
| 1 | **Logistic Regression L2** | Si `log_budget` domine, la LR capte ça parfaitement. |
| 2 | **Decision Tree** (`max_depth=3`) | Probable règle "si budget > X et genre dans {sci-fi, fantasy}" très lisible. |
| 3 | **Random Forest** (`max_depth=5`) | Robustesse, capture interaction budget × genre. |
| 4 | **XGBoost** (`max_depth=3`, `lr=0.05`) | Référence boosting, bridé. |
| 5 | **LightGBM** | Comparable, plus rapide. |


### 12.4 Pari théorique : **XGBoost (max_depth=3, lr=0.05, n_estimators=300, early stopping)**


**Pourquoi** : le signal théorique est dominé par `budget` mais avec des **interactions non triviales budget × genre × époque** (un budget de 100M en 2003 ≠ 100M en 2023). XGBoost capte ces interactions sans engineering manuel, et la régularisation L2 native + early stopping protège contre l'overfit malgré n=101.

**Plan B** : LR avec `log_budget_normalized_by_year` (z-score par décennie cf. session 03) — si la normalisation explicite suffit, la LR est imbattable en interprétabilité.


### 12.5 Feature engineering ciblé


| Feature | Recette | Pourquoi |
|---|---|---|
| `log_budget` | `log1p(budget)`, NaN si budget=0 | Distribution log-normale ; **prédicteur N°1**. |
| `budget_z_by_decade` | z-score par décennie | Normalise l'inflation et les changements de coûts (session 03). |
| `genre_is_vfx_heavy` | `genres` ∩ {Sci-Fi, Fantasy, Action, Adventure} | Filtre genre clairement déterministe. |
| `log_revenue` | `log1p(revenue)` | Coréférent du budget. |
| `is_franchise_first` | proxy : pas de chiffre dans le titre + keywords ne contiennent pas "sequel" | Effet "première impression" (Avatar, Inception). |
| `runtime_minutes` | direct | Films VFX-heavy sont plus longs. |



## 13. Meilleur Scénario Original — *Best Writing (Original Screenplay)*

**Échantillon** : n = 131, winners = 27, base-rate = 20.6 %.

### 13.1 Cadrage du sous-problème

**La seule catégorie où le contenu narratif (texte) est *directement* l'objet du prix**. Le signal devrait donc venir massivement de `overview`, `keywords`, `tagline`. Mais **attention** : nous n'avons pas le scénario lui-même, seulement un synopsis de 1-3 phrases — c'est un proxy faible. Le signal reste majoritairement porté par les features film et historiques (auteurs récurrents : Sorkin, Coen brothers, Tarantino, Anderson).
Original vs Adapted : on ne se concentre que sur l'Original → les films sont plus souvent indés / auteurs.


### 13.2 Signaux théoriques attendus

- **TF-IDF des `keywords`** : "satire", "social commentary", "non-linear narrative" sur-représentés chez les gagnants.
- **Historique du writer** (à dériver via `writers` nconsts).
- **Genre** : drama, comedy, dark comedy.
- **`is_indie`** : films à budget faible sur-représentés (Birdman, Get Out, EEAAO).
- **Co-nomination Best Picture** : signal fort.


### 13.3 Cinq modèles candidats


| # | Modèle | Justification |
|---|---|---|
| 1 | **TF-IDF(keywords + overview) + Logistic Regression L2** | **Le pipeline NLP-classique** : sparse linear est connu pour battre les modèles complexes sur petit corpus. |
| 2 | **Logistic Regression L2** (numérique seul) | Baseline sans texte pour mesurer le gain réel du texte. |
| 3 | **Random Forest** (`max_depth=6`) avec features texte réduites en SVD(20) | Ensemble robuste sur features mixtes. |
| 4 | **XGBoost** (`max_depth=4`) avec mêmes features texte | Référence boosting. |
| 5 | **Stacking** (TF-IDF+LR + XGBoost(numérique) → LR meta) | **Particulièrement adapté ici** : un modèle voit le texte, l'autre voit les chiffres ; leurs erreurs sont structurellement différentes (session 04). |


### 13.4 Pari théorique : **Stacking [ TF-IDF+LR (sur keywords⊕overview)  ⊕  XGBoost (numérique) ] → LR meta**


**Pourquoi** : c'est **la seule des 9 catégories où le stacking est théoriquement justifié de bout en bout** (session 04 : le stacking ne paie que si les modèles voient des aspects différents de la donnée). Ici, TF-IDF+LR capte la signature lexicale du scénario, et XGBoost capte la signature numérique du film/writer — leurs erreurs portent sur des films différents.

**Plan B** : TF-IDF + LR seul si le gain du stacking n'est pas significatif (peut arriver avec n=131). Plan C : LR numérique pure si le synopsis s'avère trop court pour porter un signal stable.


### 13.5 Feature engineering ciblé


| Feature | Recette | Pourquoi |
|---|---|---|
| `keywords_text` | `' '.join(keywords)` | Concaténation pour TF-IDF. |
| `overview` | direct | Texte du synopsis. |
| `text_combined` | `keywords_text + ' ' + overview` | Source de TF-IDF principale (max_features=300, ngram=(1,2)). |
| `text_svd` | `TruncatedSVD(20).fit_transform(tfidf)` | Réduction dim pour modèles non linéaires (session 07). |
| `writer_prior_noms` | parse `writers` pipe-separated, cumcount shift | Sorkin/Coen effect. |
| `is_indie` | budget dans le tercile bas de l'année | Pattern indé fort. |
| `genre_is_drama_or_comedy` | parse genres | Genre dominant. |
| `film_is_BP_nominee` | jointure | Signal général. |



## 14. Tableau de synthèse

| Catégorie | n | base-rate | Top-pick théorique | Modèle de contrôle | Modèle ambitieux |
|---|---|---|---|---|---|
| Meilleur Acteur | 135 | 20.0 % | **LR L2** + features historiques | RF (`max_depth=5`) | Stacking LR+XGBoost |
| Meilleure Actrice | 134 | 20.2 % | **LR ElasticNet** + `age_bin` | RF | Stacking LR+RF |
| Meilleur Acteur SR | 128 | 18.8 % | **Random Forest** | AdaBoost stumps | Stacking RF+AdaBoost |
| Meilleure Actrice SR | 133 | 19.5 % | **LR L2** + interactions | Decision Tree (présentation) | Stacking LR+RF |
| **Meilleur Film** | 201 | 13.4 % | **XGBoost** + TF-IDF/SVD keywords | Random Forest | Stacking LR+XGBoost+LGBM |
| Meilleur Réalisateur | 128 | 18.8 % | **LightGBM** (`directors` catégoriel) | LR + target-encoding | Stacking LR+LGBM |
| Meilleur Film Animé | 104 | 23.1 % | **Random Forest** (léger) | Decision Tree `max_depth=4` | LightGBM (si `production_company` ajouté) |
| Meilleurs Effets Visuels | 101 | 25.7 % | **XGBoost** (`max_depth=3`) | LR avec `log_budget` | Decision Tree (présentation) |
| Meilleur Scénario Original | 131 | 20.6 % | **Stacking** (TF-IDF+LR ⊕ XGBoost) | TF-IDF+LR seul | LightGBM + SVD |

**Lecture pédagogique du tableau** :
- Trois familles dominent — **LR régularisée** (régime petit-n, signal majoritairement linéaire), **RF/XGBoost** (signal interactif, signaux multiples), **Stacking** (catégories à hétérogénéité de signal type Best Picture/Screenplay).
- **AdaBoost** est le grand absent : utile en cours pour la pédagogie du boosting, mais en pratique XGBoost/LightGBM sont meilleurs **partout** sur ce problème.
- **Decision Tree** apparaît comme modèle de **présentation** (interprétabilité visuelle, plot_tree() session 05), pas comme champion attendu.
- **LightGBM** brille spécifiquement quand on a une **variable catégorielle haute-cardinalité** (Director, ou un futur `production_company` pour l'animation).


## 14b. Lecture cinéphile des paris théoriques — *pourquoi tel modèle pour telle catégorie*

Les paris du §14 sont défendus mathématiquement ci-dessus. Mais ce projet est **Applied ML for *Business***, et le business ici est le cinéma : un modèle doit aussi se justifier par la *connaissance métier*. Reprise catégorie par catégorie, avec les patterns d'Académie qui motivent vraiment le choix.

### Catégories où la baseline restera difficile à battre — c'est *attendu*

- **Meilleurs Effets Visuels** → la branche VFX vote pour le film qui *montre* le plus de technique (*Avatar*, *Gravity*, *Dune*, *Interstellar*). Or ces films sont nominés dans 6–10 catégories techniques en parallèle, donc `film_n_total_noms` est un *quasi-oracle*. Notre features-set (`budget_bin`, `genre_is_vfx_heavy`) corrèle à ~1 avec ce signal → le ML ne pourra pas faire mieux sans introduire de bruit. **Plafond intrinsèque ~75 %**.
- **Meilleur Film Animé** → catégorie créée en 2001, dominée par le **monopole Pixar/Disney** (~70 % des wins). Les exceptions sont rares et identifiables à l'œil (*Spirited Away*, *Spider-Verse*, *Pinocchio* de Del Toro). La règle simple « le film d'animation avec le plus de nominations cross-catégorie = winner » plafonnera autour de **90 %**. Le ML risque d'overfit sur n=104.
- **Meilleur Acteur dans un Second Rôle** → c'est **le** prix « it's about time » de l'Académie (Plummer à 82 ans, J.K. Simmons après 35 ans, Ke Huy Quan après 40 ans, Mahershala Ali ×2, Christopher Walken, Alan Arkin). La feature `n_prior_noms` à elle seule prédit ~55 % des vainqueurs. *Tout modèle ML qui ne respecte pas cette structure va dégrader la performance.*

### Catégories où le ML *doit* battre la baseline — et pourquoi

- **Meilleure Actrice dans un Second Rôle** → catégorie de la **révélation** (Anna Paquin à 11 ans, Jennifer Hudson 1er film, Lupita Nyong'o 1er film hollywoodien, Mo'Nique, Mikey Madison). Pattern *inverse* du Supporting Actor → `n_prior_noms` est anti-prédictif, et les features `is_first_nomination` × `is_indie` (`breakthrough_in_indie`) doivent prendre le relais. C'est *exactement* ce que le pari Decision Tree / LR L2 cherche à capter.
- **Meilleur Scénario Original** → la Writers Branch vote sur thématiques (drames sociaux, satires, *high-concepts* : *Get Out*, *Parasite*, *Promising Young Woman*, *Anatomy of a Fall*). Le **TF-IDF sur `overview + keywords`** apporte ici un vrai signal sémantique. Plus l'interaction `has_director_nom × has_acting_nom` (un scénario *triple-nominé* écrase) → c'est le terrain de jeu du Stacking / LightGBM.
- **Meilleur Film** → le signal-roi est `n_other_noms` (consensus industrie). Les BP winners sont **quasi-toujours co-nominés en Réalisation** (exceptions historiques : *Argo*, *Green Book*, *CODA*) et sortent en awards-season (octobre-décembre). LR L2 + ces features = modèle interprétable où chaque coefficient se justifie par un pattern Oscar connu.
- **Meilleur Réalisateur** → dans **19 cas sur 25** sur la période 2000–2025, le winner Director est aussi le réalisateur du BP. Les splits BP/Director sont l'exception (Soderbergh/Scott 2001, Polanski/Marshall 2003, Ang Lee ×2 pour des films *non-BP*, *Roma*/*La La Land*/*Parasite*). RF avec `film_n_total_noms` + historique personne capte bien.

### Catégories au plafond *politique*

- **Meilleur Acteur en Premier Rôle** → la catégorie la plus *campagnée* d'Hollywood (Weinstein s'y est bâti). Beaucoup de wins « hors-modèle » :
  - **DiCaprio 2016** (*The Revenant*) → vote « finally » → `is_overdue` capte.
  - **Will Smith 2022** (*King Richard*) → win le soir du slap, dynamique extra-cinéma invisible dans nos données.
  - **Brendan Fraser 2023** (*The Whale*) → comeback story émotionnelle.
  - **Anthony Hopkins 2021** (*The Father*) vs Boseman post-mortem → surprise.
  - **Adrien Brody 2025** (*The Brutalist*) → second win 22 ans après *Le Pianiste*.
  
  → Notre features-set sans signaux *precursor* (SAG/Globe/BAFTA) ni intensité de campagne FYC est intrinsèquement plafonné. **35–40 % top-1 est probablement notre plafond honnête**.
- **Meilleure Actrice en Premier Rôle** → la course la plus *consensuelle*. Sur les 10 dernières années, le **circuit SAG → Globe → Oscar** s'aligne fortement (Blanchett, Moore, Larson, Stone ×2, McDormand ×2, Colman, Zellweger, Chastain, Yeoh). Plus métrique-driven → XGBoost + features `_rel` doit naturellement bien faire (lift attendu +8–10 pts).

### Position défendue à la soutenance

Sur les 9 catégories, on **assume de garder 3 baselines en production** (Actor Supporting, Animated, VFX) plutôt que de forcer un ML pour des raisons techniques. Justification métier : dans ces 3 catégories, la heuristique « buzz / monopole / consécration vétérane » *est* le modèle optimal — c'est un pattern Oscar bien identifié, pas un artefact de petit échantillon. **Forcer un XGBoost à 30 features là où une heuristique à 1 feature est correcte serait du cargo-cult ML**. L'interprétabilité du choix prime sur la métrique.

## 15. Pièges & garde-fous (à transmettre à celui qui code)

1. **Anti-leakage temporel (session 02, critique)** : toutes les features "historiques nominée/réalisateur" doivent être calculées avec un `groupby(...).cumcount().shift(1)`. Vérifier sur un cas connu : pour Meryl Streep en 2012 (year=2012 = ceremony for 2011 films), `n_prior_wins` doit être 2, pas 3.

2. **Anti-leakage cross-catégorie** : `film_is_BP_nominee`, `film_n_total_noms` peuvent être calculés pour toutes les lignes (puisque la cible est `winner` par catégorie, pas "le film est nominé"). Mais **attention au sous-cas Best Picture** : la feature `film_is_BP_nominee` serait toujours True pour cette catégorie → la coder différemment (par exemple `n_other_noms` = nominations dans les autres catégories que la catégorie cible).

3. **Split CV correct** : `GroupKFold(n_splits=5, groups=df.year)`. **Jamais** de split aléatoire ligne-par-ligne, sinon la même année est dans train et test → leakage des co-nominations.

4. **Ne pas appliquer SMOTE** : 24-27 winners par catégorie est suffisant ; SMOTE génère du synthétique non représentatif (cf. session 01 sur l'importance du contexte business).

5. **TF-IDF sur 100 lignes — précautions** :
   - `max_features ≤ 300`, `min_df ≥ 2`, `max_df ≤ 0.9`.
   - Toujours réduire en TruncatedSVD(20-30) avant de feeder dans un modèle non linéaire.
   - **Ne pas réutiliser** un vectoriseur fitté sur le full dataset — refitter à chaque fold.

6. **Métrique de présentation** : tout reporter en **top-1 accuracy par groupe (catégorie, année)**, c'est la métrique métier. Les PR-AUC / log-loss sont des sanity-checks internes.

7. **Calibration** : si l'app Streamlit affiche des probabilités, calibrer (Platt scaling / isotonic) en CV avant de présenter — un modèle à 90 % de confiance qui se trompe 30 % du temps détruira la crédibilité du projet.

8. **Drift temporel (PSI session 02)** : avant de finaliser, calculer le PSI des features-clés entre 2000-2015 et 2016-2025. Le profil-type du gagnant a changé (montée des films internationaux, plus de 10 nominés en Best Picture). Si PSI > 0.2, envisager une feature `decade` ou un poids exponentiel sur les années récentes.

9. **Sample size warning** : 100-130 lignes c'est *vraiment* peu. Tout écart de performance < 3 points entre deux modèles est probablement dans le bruit de CV. Toujours rapporter la **moyenne ± écart-type sur 5 folds**, pas un score unique.

10. **Garder le notebook EDA propre** : ce document est une hypothèse de modélisation. Le notebook d'expérimentation (à venir) devra distinguer clairement (i) features computées une fois pour toutes (ii) features dépendantes du fold (target encoding, TF-IDF).


## 16. Annexe — squelette de code recommandé (pour l'équipe d'implémentation)

```python
from sklearn.model_selection import GroupKFold
from sklearn.metrics import average_precision_score, log_loss
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
import xgboost as xgb
import lightgbm as lgb
import numpy as np

def evaluate_by_group(df_cat, X, y, model, groups):
    \"\"\"Top-1 accuracy par (category, year) sur 5 folds GroupKFold.\"\"\"
    gkf = GroupKFold(n_splits=5)
    accs, aps = [], []
    for tr, te in gkf.split(X, y, groups=groups):
        model.fit(X[tr], y[tr])
        proba = model.predict_proba(X[te])[:, 1]
        # top-1 par année sur le fold de test
        df_te = df_cat.iloc[te].assign(proba=proba)
        top1 = (df_te.groupby('year')
                       .apply(lambda g: g.loc[g['proba'].idxmax(), 'winner'])
                       .mean())
        accs.append(top1)
        aps.append(average_precision_score(y[te], proba))
    return np.mean(accs), np.std(accs), np.mean(aps)

# Boucle catégorie → modèle (squelette)
for cat in TARGETS:
    df_cat = df[df.category == cat].reset_index(drop=True)
    y = df_cat['winner'].astype(int).values
    groups = df_cat['year'].values
    X = build_features(df_cat)  # ← à implémenter selon §4 et §5-13

    models = {
        'lr_l2'   : Pipeline([('sc', StandardScaler()),
                              ('lr', LogisticRegression(C=1.0, class_weight='balanced',
                                                        max_iter=2000, random_state=42))]),
        'rf'      : RandomForestClassifier(n_estimators=300, max_depth=5,
                                           min_samples_leaf=5, class_weight='balanced',
                                           random_state=42),
        'xgboost' : xgb.XGBClassifier(max_depth=4, learning_rate=0.05, n_estimators=400,
                                       scale_pos_weight=(y==0).sum()/(y==1).sum(),
                                       random_state=42, eval_metric='logloss'),
        'lightgbm': lgb.LGBMClassifier(num_leaves=15, min_data_in_leaf=10,
                                        learning_rate=0.05, n_estimators=400,
                                        class_weight='balanced', random_state=42),
        # stacking : voir StackingClassifier(cv=5) — session 04
    }

    for name, m in models.items():
        acc, std, ap = evaluate_by_group(df_cat, X, y, m, groups)
        print(f'{cat:48s} {name:10s} top1={acc:.3f}±{std:.3f}  PR-AUC={ap:.3f}')
```

> Le pipeline complet, avec construction des features historiques et TF-IDF par fold, sera fait dans un autre notebook par l'équipe. Ce document fixe **quoi** essayer et **pourquoi**, pas **comment** le coder en intégralité.
